In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Entradas y salidas de Estimator

<Accordion>
<AccordionItem title="Versiones de paquetes">

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Esta página ofrece una descripción general de las entradas y salidas del primitivo Estimator de Qiskit Runtime, que ejecuta cargas de trabajo en recursos de cómputo de IBM Quantum&reg;. Estimator te permite definir eficientemente cargas de trabajo vectorizadas usando una estructura de datos llamada [**Bloque Unificado Primitivo (PUB)**](/guides/primitive-input-output#pubs). Se usan como entradas al método [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) del primitivo Estimator, que ejecuta la carga de trabajo definida como un trabajo. Luego, después de que el trabajo haya completado, los resultados se devuelven en un formato que depende tanto de los PUBs utilizados como de las opciones de runtime especificadas desde el primitivo.
## Entradas
Cada PUB está en este formato:

(`<circuit único>`, `<uno o más observables>`, `<valores de parámetros opcionales>`, `<precisión opcional>`),

Los `valores de parámetros` opcionales pueden ser una lista o un solo parámetro. Los elementos de los observables y los valores de parámetros se combinan siguiendo las reglas de difusión de NumPy como se describe en el tema [Entradas y salidas de primitivos](primitive-input-output#broadcasting-rules), y se devuelve una estimación del valor de expectación para cada elemento de la forma difundida.

> **Note:** Si la entrada contiene mediciones, se ignoran.

Para el primitivo Estimator, un PUB puede contener como máximo cuatro valores:
- Un único `QuantumCircuit`, que puede contener uno o más objetos [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Una lista de uno o más observables, que especifican los valores de expectación a estimar, organizados en un array (por ejemplo, un solo observable representado como un array 0-d, una lista de observables como un array 1-d, etc.). Los datos pueden estar en cualquiera de los formatos `ObservablesArrayLike` como `Pauli`, `SparsePauliOp`, `PauliList` o `str`.
   > **Note:** - Los observables que conmutan **en el mismo PUB** se agrupan usando [este método](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Los observables que conmutan en diferentes PUBs, aunque tengan el mismo circuit, no se estiman usando la misma medición. Cada PUB representa una base diferente para la medición y, por lo tanto, se requieren mediciones separadas para cada PUB.
>    - Para asegurarte de que los observables que conmutan se estimen usando la misma medición, agrúpalos dentro del mismo PUB.
- Una colección de valores de parámetros para vincular al circuit. Esto se puede especificar como un único objeto similar a un array donde el último índice es sobre los objetos `Parameter` del circuit, u omitido (o equivalentemente, establecido en `None`) si el circuit no tiene objetos `Parameter`.
- (Opcionalmente) Una precisión objetivo para los valores de expectación a estimar
---

El siguiente código demuestra un conjunto de ejemplo de entradas vectorizadas al primitivo `Estimator` y las ejecuta en un backend de IBM&reg; como un único objeto `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Salidas
Después de que uno o más PUBs son enviados a una QPU para su ejecución y un trabajo completa exitosamente, los datos se devuelven como un objeto contenedor [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) al que se accede llamando al método `RuntimeJobV2.result()`.

El `PrimitiveResult` contiene una lista iterable de objetos [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) que contienen los resultados de ejecución para cada PUB.

Cada elemento de esta lista corresponde a cada PUB enviado al método `run()` del primitivo (por ejemplo, un trabajo enviado con 20 PUBs devolverá un objeto `PrimitiveResult` que contiene una lista de 20 objetos `PubResult`, uno correspondiente a cada PUB).

Cada `PubResult` para el primitivo Estimator contiene al menos un array de valores de expectación (`PubResult.data.evs`) y desviaciones estándar asociadas (ya sea `PubResult.data.stds` o `PubResult.data.ensemble_standard_error` dependiendo del `resilience_level` utilizado), pero puede contener más datos dependiendo de las opciones de mitigación de errores especificadas.

Cada objeto `PubResult` posee tanto un atributo `data` como un atributo `metadata`.
    - El atributo `data` es un [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) personalizado que contiene los valores de medición reales, las desviaciones estándar, etc.
    - El `DataBin` tiene varios atributos dependiendo de la forma o estructura del PUB asociado, así como de las opciones de mitigación de errores especificadas por el primitivo utilizado para enviar el trabajo (por ejemplo, [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) o [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - El atributo `metadata` contiene información sobre las opciones de runtime y mitigación de errores utilizadas (explicado más adelante en la sección [Metadatos del resultado](#result-metadata) de esta página).

El siguiente es un esquema visual de la estructura de datos de `PrimitiveResult` para la salida de Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

En pocas palabras, un solo trabajo devuelve un objeto `PrimitiveResult` y contiene una lista de uno o más objetos `PubResult`. Estos objetos `PubResult` luego almacenan los datos de medición para cada PUB que fue enviado al trabajo.

El siguiente fragmento de código describe el formato `PrimitiveResult` (y el `PubResult` asociado) para el trabajo creado anteriormente.

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
